# E9 fixed models — 2026-10-24 14:43
Intent: implement one history-only LightFM candidate generator and one fixed binary LightGBM reranker. Synthetic checks cover eligibility, deduplication, the 500-row cap, source interleaving, uncapped baselines, cold starts, empty inputs, and train-only categorical vocabularies.
Method: fixed WARP32/15 epochs and fixed binary LightGBM250; no validation-label access. Existing .venv-exp5 runtime, with cached OpenMP located through DYLD_LIBRARY_PATH; no dependency mutations. LightFM reports no OpenMP support and therefore executes single-threaded.

In [1]:
import os, sys, pathlib, numpy as np, pandas as pd
root = pathlib.Path.cwd()
if not (root / 'artifacts/scripts/e9_models.py').exists(): root = root.parent
sys.path.insert(0, str(root / 'artifacts/scripts'))
import e9_models as m
print('Interpreter:', sys.executable)
print('DYLD_LIBRARY_PATH:', os.environ.get('DYLD_LIBRARY_PATH'))
compile((root / 'artifacts/scripts/e9_models.py').read_text(), 'e9_models.py', 'exec')
history = pd.DataFrame([('a', 1, .8), ('a', 1, .8), ('a', 2, .8), ('a', 2, .1), ('a', 3, .2), ('other', 4, .9), ('other', 900, .9)], columns=['user_id','tv_show_id','user_watch_pct'])
future = pd.DataFrame({'tv_show_id': list(range(1, 621)) + [1]})
captured = []
Original = m.LightFM
class RecordingLightFM(Original):
    def fit(self, matrix, **kwargs):
        captured.append((self, matrix.copy()))
        return super().fit(matrix, **kwargs)
m.LightFM = RecordingLightFM
candidates = m.generate_candidates(history, future, ['a','cold','a'])
m.LightFM = Original
lf, matrix = captured[0]
assert matrix.shape == (3, 621) and matrix.nnz == 4 and np.all(matrix.data == 1)
assert len(candidates) == 1000 and not candidates.duplicated(['user_id','tv_show_id']).any()
assert candidates.tv_show_id.between(1,620).all() and candidates.user_id.map(type).eq(str).all()
assert list(candidates.columns) == m.COLUMNS
assert candidates.attrs['repeat_top5'] == {'a': [1,2,3], 'cold': []}
for user, group in candidates.groupby('user_id', sort=False):
    assert group.candidate_rank.tolist() == list(range(1,501))
    vectors = lf.item_embeddings[:620].copy()
    vectors[[i not in {0,1,3} for i in range(620)]] = 0
    scores = lf.item_biases[:620].copy()
    if user == 'a': scores += vectors @ lf.user_embeddings[0] + lf.user_biases[0]
    order = np.lexsort((np.arange(1,621), -scores)) + 1
    assert candidates.attrs['lightfm_top5'][user] == order[:5].tolist()
    keys = {int(s): (r,1,int(s)) for r,s in enumerate(order[:500],1)}
    for r,s in enumerate([1,2,3] if user == 'a' else [],1): keys[s] = min(keys.get(s,(r,0,s)), (r,0,s))
    assert group.tv_show_id.tolist() == sorted(keys, key=keys.get)[:500]
    np.testing.assert_allclose(group.lightfm_score, scores[group.tv_show_id.to_numpy()-1])
    assert group.lightfm_rank.tolist() == [int(np.where(order == s)[0][0]+1) for s in group.tv_show_id]
assert m.generate_candidates(history, future.iloc[:0], ['a']).empty
cold = m.generate_candidates(history.iloc[:0], future.iloc[:3], ['cold'])
assert cold.tv_show_id.tolist() == [1,2,3] and cold.lightfm_score.eq(0).all()
rng = np.random.default_rng(42)
train = pd.DataFrame({c: rng.normal(size=240) for c in m.FEATURES})
for c in m.CATEGORICAL: train[c] = ['known','other'] * 120
train['label'] = (train.user_show_freq_05 > 0).astype(int)
train['user_id'], train['tv_show_id'], train['candidate_rank'] = 'excluded', 99, 99
valid = train.iloc[:8].drop(columns='label').copy()
for c in m.CATEGORICAL: valid[c] = 'unseen'
probabilities, model = m.fit_predict(train, valid)
assert probabilities.shape == (8,) and np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
assert model.feature_name_ == m.FEATURES
assert all(v == ['known','other'] for v in model.e9_category_vocab_.values())
assert model.booster_.pandas_categorical == [['known','other']] * 4
assert pd.Categorical(valid.category.where(valid.category.isin(model.e9_category_vocab_['category'])), categories=model.e9_category_vocab_['category']).isna().all()
print('PASS compile; binary distinct history matrix shape=(3,621), nnz=4')
print('PASS candidates: 7 history rows + 620 eligible shows -> 1000 rows / 2 users; deterministic source ordering, exact scores/ranks, full baselines, cold and empty paths')
print('PASS reranker: 240 train rows -> 8 probabilities; 19 allowlisted features, train-only categories, no validation label column')
print('Probability range:', float(probabilities.min()), float(probabilities.max()))


/Users/davidkk/Kaggle_Case/.venv-exp5/lib/python3.12/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


Interpreter: /Users/davidkk/Kaggle_Case/.venv-exp5/bin/python
DYLD_LIBRARY_PATH: /Users/davidkk/.cache/uv/archive-v0/SejYHQMpdHzmkdZM/sklearn/.dylibs
PASS compile; binary distinct history matrix shape=(3,621), nnz=4
PASS candidates: 7 history rows + 620 eligible shows -> 1000 rows / 2 users; deterministic source ordering, exact scores/ranks, full baselines, cold and empty paths
PASS reranker: 240 train rows -> 8 probabilities; 19 allowlisted features, train-only categories, no validation label column
Probability range: 7.74348325162688e-06 0.9999940004216199


Result interpretation: all synthetic assertions and compilation passed (exit 0). Seven history rows and 620 eligible shows produced 500 candidates each for two evaluation users. The binary model trained on 240 rows and returned eight finite probabilities using exactly 19 features; validation had no label column and contained unseen categorical values.
Cold-start users use actual fitted item biases only; items with no positive history have their scoring dot-product zeroed, retaining actual fitted item biases. Baseline maps preserve full eligible-catalog LightFM top5 and history-only repeat top5.
Execution command from repository root:
```sh
DYLD_LIBRARY_PATH=/Users/davidkk/.cache/uv/archive-v0/SejYHQMpdHzmkdZM/sklearn/.dylibs .venv-exp5/bin/python - <<'PY'
import json
from pathlib import Path
p = Path('experiments/2026-10-24-1443-e9-models.ipynb')
notebook = json.loads(p.read_text())
exec(compile(''.join(notebook['cells'][1]['source']), str(p), 'exec'))
print('Implementation lines:', len(Path('artifacts/scripts/e9_models.py').read_text().splitlines()))
PY
```
Failed attempt: initial direct LightGBM import exited 1 because libomp.dylib was unresolved. The existing cached library resolved it without dependency changes. The first smoke passed with a pandas future-compatibility warning; unknown categories are now explicitly masked before categorical conversion and the repeated smoke passed without that warning. No reverted changes. No full-data training; no recommendation-quality claim.